# 03 - Exploratory Data Analysis

Core exploratory analysis with business-relevant visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (10, 6)

DATA_PATH = Path("../data/processed/online_retail_cleaned.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["InvoiceDate"])
df.head()

## 1. Revenue and Time Trends

### Monthly Revenue

In [ ]:
df_sales = df[df["OrderStatus"] == "normal"].copy()
monthly_revenue = df_sales.groupby(df_sales["InvoiceDate"].dt.to_period("M"))["Sales"].sum()
monthly_revenue.index = monthly_revenue.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_revenue.index, monthly_revenue.values, color="#3498db", linewidth=2)
ax.fill_between(monthly_revenue.index, monthly_revenue.values, alpha=0.3)
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.set_title("Monthly Revenue Trend")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/monthly_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** The monthly revenue chart reveals seasonal patterns. Peaks often align with holiday periods (e.g., Christmas). Use this to plan inventory and marketing campaigns.

### Hour vs Weekday Heatmap

In [ ]:
heatmap_data = df_sales.pivot_table(
    values="Sales", index="Hour", columns="DayOfWeek", aggfunc="sum"
).fillna(0)
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
heatmap_data.columns = [day_names[i] for i in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=False, fmt=".0f", cmap="YlOrRd", ax=ax)
ax.set_xlabel("Day of Week")
ax.set_ylabel("Hour")
ax.set_title("Sales Intensity: Hour vs Day of Week")
plt.tight_layout()
plt.savefig("../reports/figures/hour_weekday_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** Identify peak shopping hours and days. Schedule customer support and promotions accordingly.

## 2. Product Insights

### Top 20 Best-Selling Products

In [ ]:
top_products = (
    df_sales.groupby(["StockCode", "Description"])["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 8))
top_products.plot(kind="barh", ax=ax, color="#2ecc71")
ax.set_xlabel("Quantity Sold")
ax.set_ylabel("")
ax.set_title("Top 20 Best-Selling Products")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../reports/figures/top_products.png", dpi=150, bbox_inches="tight")
plt.show()

### Price Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_sales["UnitPrice"].clip(upper=50), bins=50, color="#9b59b6", edgecolor="white", alpha=0.7)
ax.set_xlabel("Unit Price")
ax.set_ylabel("Count")
ax.set_title("Price Distribution (capped at 50 for visibility)")
plt.tight_layout()
plt.savefig("../reports/figures/price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Price vs Quantity Scatter

In [ ]:
sample = df_sales.sample(min(5000, len(df_sales)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(sample["UnitPrice"].clip(upper=50), sample["Quantity"].clip(upper=100), alpha=0.3, s=10)
ax.set_xlabel("Unit Price")
ax.set_ylabel("Quantity")
ax.set_title("Price vs Quantity (sample, clipped for visibility)")
plt.tight_layout()
plt.savefig("../reports/figures/price_vs_quantity.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Customer Behavior

### Revenue by Customer Segment (RFM Preview)

In [ ]:
customer_revenue = df_sales.groupby("CustomerID")["Sales"].sum().sort_values(ascending=False)
top_customers = customer_revenue.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_customers.plot(kind="bar", ax=ax, color="#e74c3c")
ax.set_xlabel("Customer ID")
ax.set_ylabel("Total Revenue")
ax.set_title("Top 15 Customers by Revenue")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/top_customers.png", dpi=150, bbox_inches="tight")
plt.show()

### Average Basket Size by Country

In [ ]:
basket_by_country = (
    df_sales.groupby(["InvoiceNo", "Country"])["Sales"].sum()
    .reset_index()
    .groupby("Country")["Sales"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
basket_by_country.plot(kind="bar", ax=ax, color="#1abc9c")
ax.set_xlabel("Country")
ax.set_ylabel("Average Basket Size (Revenue)")
ax.set_title("Average Basket Size by Country (Top 15)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/basket_by_country.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Cross-Country Analysis

### UK vs Rest of World

In [ ]:
df_sales["Region"] = np.where(df_sales["Country"] == "United Kingdom", "UK", "Rest of World")
region_revenue = df_sales.groupby("Region")["Sales"].sum()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(region_revenue, labels=region_revenue.index, autopct="%1.1f%%", colors=["#3498db", "#95a5a6"])
ax.set_title("Revenue: UK vs Rest of World")
plt.tight_layout()
plt.savefig("../reports/figures/uk_vs_row.png", dpi=150, bbox_inches="tight")
plt.show()

### Top Countries by Revenue

In [ ]:
country_revenue = df_sales.groupby("Country")["Sales"].sum().sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 6))
country_revenue.plot(kind="bar", ax=ax, color="#f39c12")
ax.set_xlabel("Country")
ax.set_ylabel("Total Revenue")
ax.set_title("Top 12 Countries by Revenue")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/top_countries.png", dpi=150, bbox_inches="tight")
plt.show()